# AND ERA5-Land Annual Test

This notebook launches a focused annual ERA5-Land rerun for the Andrews (AND/HJA) watersheds. It uses the same extraction code as the main workflow and keeps only the AND watersheds before exporting.

This is the corrected small-watershed run: it calculates each value from the watershed polygon first, then retries the same polygon at a finer scale if the first result is blank. It does not fill blank values from the watershed centroid. Rows filled by the retry are marked with `used_fine_scale_fallback`.

The default run is 2001-2023 so it overlaps with the reference MODIS and climate-driver columns. It writes one CSV per year to the Google Drive folder printed below.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess

# Clone the workflow the first time; refresh it on later notebook runs.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

import sys
import importlib

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src.gee_spatial"):
        del sys.modules[module_name]
importlib.invalidate_caches()


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

DRIVE_EXPORT_FOLDER = "and_tiny_watersheds_gee_exports_snow8day_2001_2023"

assets.setdefault("exports", {}).update(
    {
        "destination": "drive",
        "drive_folder": DRIVE_EXPORT_FOLDER,
    }
)

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed asset:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder"))


## Edit Run Settings

Change the settings in the next cell before choosing **Runtime > Run all**. Defaults run the corrected AND tiny-watershed test for 2001-2023. Use `SITE_IDS` or `MAX_TASKS_TO_LAUNCH` for a smaller smoke test.


In [ ]:
# Output names include RUN_LTER_LABEL and RUN_LABEL.
RUN_LTER_LABEL = "AND"
RUN_LABEL = "fine_scale"

# Years are inclusive. Use YEARS_TO_SKIP for partial reruns after some exports finish.
START_YEAR = 2001
END_YEAR = 2023
YEARS_TO_SKIP = []

# Leave as None for the full test. Set to 1 or 2 for a quick smoke test.
MAX_TASKS_TO_LAUNCH = None

# Run all launches tasks and prints one status snapshot. Set TRUE to keep Colab polling until every task finishes.
WAIT_FOR_TASKS = False

# Leave SITE_IDS empty for all AND watersheds, or paste a short site_id list.
SITE_IDS = []

# Product options: precip, temp, evapotrans, potential_evap, snow_cover, snow_water_equiv.
PRODUCTS = [
    "precip",
    "temp",
    "evapotrans",
    "potential_evap",
    "snow_cover",
    "snow_water_equiv",
]


In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()

# Use the column names from the uploaded watershed asset, even if capitalization changed.
lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])
site_id_column = choose_property_name(watersheds, "site_id")
lter_values = watersheds.aggregate_array(lter_column).distinct().sort().getInfo()
# Find the asset's exact spelling of the requested LTER before filtering rows.
and_lter_value = next(
    (
        value for value in lter_values
        if str(value).strip().lower() == str(RUN_LTER_LABEL).strip().lower()
    ),
    None,
)

print("Asset rows:", watersheds.size().getInfo())
print("Asset columns:", property_names)
print("LTER column:", lter_column)
print("Site ID column:", site_id_column)
print("LTER values:", lter_values)

# Stop before launching anything if the uploaded asset did not contain AND rows.
if and_lter_value is None:
    raise ValueError(f"No {RUN_LTER_LABEL} value was found in the uploaded watershed asset LTER column.")

# Keep only the requested LTER rows for this focused check.
and_watersheds = watersheds.filter(ee.Filter.eq(lter_column, and_lter_value))
if SITE_IDS:
    and_watersheds = and_watersheds.filter(ee.Filter.inList(site_id_column, SITE_IDS))
selected_row_count = and_watersheds.size().getInfo()

print("Run LTER label:", RUN_LTER_LABEL)
print("Asset LTER value:", and_lter_value)
print("Site ID filters:", SITE_IDS)
print("Selected watershed rows:", selected_row_count)
print("Selected preview:")
print(and_watersheds.limit(5).getInfo())

# Stop before launching anything if the AND filter did not keep any rows.
if selected_row_count == 0:
    raise ValueError("No watershed rows were selected.")


## Launch Annual Exports

This starts one Earth Engine table export per year. To try just a few years first, change `MAX_TASKS_TO_LAUNCH` to a small number like `3`.

The output file names include `AND_fine_scale` so they are easy to separate from the earlier centroid-fallback files. Only rerun this cell if you mean to start another set of exports.


In [ ]:
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import era5_export_columns, extract_era5_land_products
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

years_to_launch = [
    year for year in range(START_YEAR, END_YEAR + 1)
    if year not in set(YEARS_TO_SKIP)
]
if MAX_TASKS_TO_LAUNCH is not None:
    years_to_launch = years_to_launch[: int(MAX_TASKS_TO_LAUNCH)]

session_started_at = utc_now()
timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_era5_land_{RUN_LTER_LABEL}_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)

selectors = era5_export_columns(products, product_names=PRODUCTS, include_snow_comparison_metric=True)
launched_tasks = []

print("Years to launch:", years_to_launch)
print("Years skipped:", YEARS_TO_SKIP)
print("Selected watershed rows:", selected_row_count)
print("Drive output folder:", assets["exports"].get("drive_folder"))
print("Export columns:", selectors)
print("Timing log:", timing_log_path)

# Launch one export per year so each CSV stays small and easy to rerun.
for year in years_to_launch:
    out_name = f"era5_land_{year}_{RUN_LTER_LABEL}_{RUN_LABEL}_watershed_extract"
    print("\nLaunching:", out_name)

    # Build the Earth Engine table for this year and this set of products.
    export_rows = extract_era5_land_products(
        products,
        and_watersheds,
        year=year,
        product_names=PRODUCTS,
        include_snow_comparison_metric=True,
    )

    # Start the Drive export. Earth Engine will run this in the background.
    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    # Save enough timing information to check progress later.
    launched_at = utc_now()
    run_row = {
        "run_name": f"era5_land_{RUN_LTER_LABEL}_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}",
        "mode": "era5_land",
        "products": PRODUCTS,
        "year": year,
        "month": None,
        "months": None,
        "period": "annual",
        "run_group": f"{RUN_LTER_LABEL}_{RUN_LABEL}",
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {
            "name": out_name,
            "task": task,
            "launched_at": launched_at,
            "timing_row": timing_row,
        }
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

print("\nLaunched tasks:", len(launched_tasks))
print("Timing log:", timing_log_path)


## Check Task Progress

Run this cell after the export cell. It checks Earth Engine once per minute until every task finishes or fails.


In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    FAILED_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60
wait_for_tasks = bool(globals().get("WAIT_FOR_TASKS", False))


def print_existing_earth_engine_tasks(name_filter=None, max_tasks=40):
    name_filter = name_filter or ""
    matches = []
    for task in ee.batch.Task.list():
        status = task.status()
        description = status.get("description", "")
        if name_filter and name_filter not in description:
            continue
        matches.append(status)

    if not matches:
        print("No Earth Engine tasks matched filter:", name_filter or "<none>")
        return

    print("Earth Engine task status snapshot for filter:", name_filter or "<none>")
    for status in matches[:max_tasks]:
        print(status.get("description", ""), status.get("state", ""), status.get("error_message", ""))
    if len(matches) > max_tasks:
        print("Additional matching tasks not shown:", len(matches) - max_tasks)


def default_task_status_filter():
    if "RUN_LTER_LABEL" in globals() and "RUN_LABEL" in globals():
        return f"{RUN_LTER_LABEL}_{RUN_LABEL}"
    return globals().get("RUN_LABEL", globals().get("active_run", ""))

def refresh_launched_tasks_once():
    now = utc_now()
    print()
    print("Task status snapshot at", now.isoformat(timespec="seconds"))
    all_done = True

    for item in launched_tasks:
        item["timing_row"] = update_task_timing_row(
            item["timing_row"],
            item["task"],
            checked_at=now,
        )
        state = item["timing_row"].get("state")
        elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
        error_message = item["timing_row"].get("error_message", "")
        print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")
        if error_message:
            print("  error:", error_message)

        if state not in DONE_STATES:
            all_done = False

    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    return all_done


if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this Colab session memory.")
    print_existing_earth_engine_tasks(default_task_status_filter())
else:
    all_done = refresh_launched_tasks_once()

    if wait_for_tasks:
        while not all_done:
            time.sleep(poll_seconds)
            all_done = refresh_launched_tasks_once()
    else:
        print()
        print("WAIT_FOR_TASKS is FALSE, so this notebook will not keep polling.")
        print("The Earth Engine exports continue server-side after launch.")
        print("Rerun this cell later, or check the Earth Engine task list, to see final status.")

    final_rows = timing_rows_from_launched_tasks(launched_tasks)
    print_timing_summary(final_rows)
    failed_rows = [row for row in final_rows if row.get("state") in FAILED_STATES]

    run_wall_clock_finished_at = utc_now()
    run_wall_clock_elapsed_min = None
    if "run_wall_clock_timer_started" in globals():
        run_wall_clock_elapsed_min = (time.perf_counter() - run_wall_clock_timer_started) / 60
    elif "run_wall_clock_started_at" in globals():
        run_wall_clock_elapsed_min = (
            run_wall_clock_finished_at - run_wall_clock_started_at
        ).total_seconds() / 60

    if failed_rows:
        if "write_run_wall_clock_summary" in globals():
            write_run_wall_clock_summary(
                status="failed",
                finished_at=run_wall_clock_finished_at,
                elapsed_min=run_wall_clock_elapsed_min,
            )
        for row in failed_rows:
            print(row.get("export_name"), row.get("state"), row.get("error_message", ""))
        raise RuntimeError(f"Earth Engine export failed for {len(failed_rows)} task(s).")

    if wait_for_tasks and all_done:
        summary_status = "completed"
    elif all_done:
        summary_status = "completed_snapshot"
    else:
        summary_status = "launched"

    if "write_run_wall_clock_summary" in globals():
        write_run_wall_clock_summary(
            status=summary_status,
            finished_at=run_wall_clock_finished_at,
            elapsed_min=run_wall_clock_elapsed_min,
        )
        print("Run timing summary:", run_timing_summary_path)

    print("Task timing log:", timing_log_path)


In [ ]:
# Optional: rerun this cell later to check matching Earth Engine tasks after reconnecting.
TASK_STATUS_FILTER = default_task_status_filter() if "default_task_status_filter" in globals() else globals().get("RUN_LABEL", globals().get("active_run", ""))

if "print_existing_earth_engine_tasks" not in globals():
    def print_existing_earth_engine_tasks(name_filter=None, max_tasks=40):
        name_filter = name_filter or ""
        matches = []
        for task in ee.batch.Task.list():
            status = task.status()
            description = status.get("description", "")
            if name_filter and name_filter not in description:
                continue
            matches.append(status)

        if not matches:
            print("No Earth Engine tasks matched filter:", name_filter or "<none>")
            return

        print("Earth Engine task status snapshot for filter:", name_filter or "<none>")
        for status in matches[:max_tasks]:
            print(status.get("description", ""), status.get("state", ""), status.get("error_message", ""))
        if len(matches) > max_tasks:
            print("Additional matching tasks not shown:", len(matches) - max_tasks)

print_existing_earth_engine_tasks(TASK_STATUS_FILTER)


## After The Exports Finish

Completed Earth Engine tasks write CSVs to the Drive export folder named `and_tiny_watersheds_gee_exports_snow8day_2001_2023`. Use the local R scripts in `workflow/` to move the CSVs into a tidy run folder and run QA or inventory steps.


By default, `WAIT_FOR_TASKS = False`, so **Runtime > Run all** launches exports and then stops after a status snapshot. The Earth Engine tasks continue server-side. Rerun the status-check cell later to confirm completion before running local R QA or inventory steps.
